In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from funciones_t6 import laplace2D, random_walk_2d
import imageio 


ModuleNotFoundError: No module named 'numba'

In [ ]:

# Semilla para numeros aleatorios
np.random.seed(1)

# Parametros caminantes
N = 100 # numero de pasos
nw = 1000 # Numero de caminantes
dr = 1  # Distancia de paso radial

dist = np.zeros([nw,N+1], dtype= float)
angulo = np.zeros([nw,N+1], dtype= float)

# Parametros difusion
D = 1
deltax = 1
deltat = deltax**2 / (4*D) /2  # estabilidad en 2D / 2

# Numero de puntos en x e y, tiempo final y puntos para ts
NX = 50
NY = 50

tf = 100.0
NT = int(100/deltat)

# Arrays simétricos (incluimos +1)
x = np.arange(-NX, NX+1, deltax)
y = np.arange(-NY, NY+1, deltax)
t = np.arange(0, (NT + 1)*deltat, deltat)

# densidad: (x, y, t)
rho = np.zeros((2*NX+1, 2*NY+1, NT+1))


# condición inicial: delta en el centro
rho[NX, NY, 0] = 1

# Calculamos la evolución temporal
ts, rho = laplace2D(T= rho, tf= tf, dt = deltat)

# Calculamos el random walk
xw, yw = random_walk_2d(nw= nw, n = N)

In [ ]:
# Calculamos el cuadrado de los desplazamientos y el desplazamiento cuadratico promedio
x2, y2 = np.sum(xw**2, 0), np.sum(yw**2, 0)

r2 = (x2 + y2)/nw
  

x_fit = np.linspace(0,N,101)
coef= np.polyfit(x_fit, r2, 1)

ajuste  = coef[0]*x_fit + coef[1]

fig4, ax4 = plt.subplots()

ax4.plot(x_fit,r2, label = 'Desplazamiento cuadrático promedio')
ax4.plot(x_fit,ajuste, label = f'y = {coef[0]:.2f}x + {coef[1]:.1f}')

ax4.legend()
ax4.grid()


In [ ]:

# Calculamos la entropia con un histograma en dos dimensiones
S = np.zeros(N+1)

for i in range(N+1):
    H, _, _ = np.histogram2d(
        xw[:, i], yw[:, i],
        bins=10,
        range=[[-NX, NX], [-NY, NY]]
    )
    
    p = H / np.sum(H)
    
    # log seguro
    S[i] = -np.sum(p * np.log(p + 1e-12))

fig3, ax3 = plt.subplots()

ax3.plot(S)

ax3.set_xlabel('Tiempo (s)')
ax3.set_ylabel('Entropía')

plt.show()
    




In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(8,4))

# Random walk
scat = ax[0].scatter([], [], s=1, alpha=0.6)
ax[0].set_xlim(-NX, NX)
ax[0].set_ylim(-NY, NY)
ax[0].set_title("Random Walk")
ax[0].axis('equal')

# Difusión
vmax = np.max(rho) / 100 
img = ax[1].imshow(
    rho[:, :, 0],
    extent=[-NX, NX, -NY, NY],
    origin='lower',
    vmin=0,
    vmax=vmax
)

cbar = fig.colorbar(img, ax=ax[1])
cbar.set_label("Densidad")

ax[1].set_title("Difusión")
ax[1].axis('equal')

ax[0].set_box_aspect(1)
ax[1].set_box_aspect(1)

# --- GENERACIÓN DE FRAMES ---
images = []

for i in range(N+1):
    # Actualizar datos
    scat.set_offsets(np.c_[xw[:, i], yw[:, i]])
    img.set_data(rho[:, :, i])

    ax[0].set_title(f"Random Walk (t={i})")
    ax[1].set_title(f"Difusión (t={i})")

    # Dibujar figura
    fig.canvas.draw()

    buffer = fig.canvas.buffer_rgba()
    image = np.asarray(buffer)
    
    images.append(image.copy())


# --- GUARDAR GIF ---
imageio.mimsave("random_walk_vs_diffusion.mp4", images, fps=10)

plt.close(fig)